# Ingesta de datos de temperaturas medias de CONAGUA
Esta libreta de Jupyter se utilizará para la ingesta y procesamiento de datos relacionados al registro de temperaturas medias en distintas estaciones metereológicas distribuidas en todo México. Los datos provienen de la base de datos de la CONAGUA. 

## Ingesta de datos

In [2]:
# Importación de librerías
import os
import time
import requests
import urllib3

# Desactivar advertencias de certificados SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configuración general y definición de rutas
base_url = (
    "https://smn.conagua.gob.mx/tools/RESOURCES/com_archivo_datos_resumenes"
)

# Carpeta destino dentro de data/raw/ subiendo desde notebooks/
main_folder = os.path.join("..", "data", "raw")
os.makedirs(main_folder, exist_ok=True)

# Cabeceras completas para autorizar la petición en CONAGUA
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    ),
    "Referer": (
        "https://smn.conagua.gob.mx/es/climatologia/temperaturas-y-lluvias/resumenes-mensuales-de-temperaturas-y-lluvias"
    ),
}

start_year = 2019
end_year = 2024  # Abarca del 2019 al 2023 inclusive


# BUCLES DE DESCARGA (AÑOS Y MESES)

for current_year in range(start_year, end_year):
  year_str = str(current_year)
  print("----------------------------------------")
  print("Descarga del año: " + year_str)
  print("----------------------------------------")

  # Carpeta anual directa: ../data/raw/temperatura_XXXX
  year_folder = os.path.join(main_folder, "temperatura_" + year_str)
  os.makedirs(year_folder, exist_ok=True)

  for month in range(1, 13):
    mes_str = str(month).zfill(2)
    file_name = year_str + mes_str + "010000TMed.csv"
    file_url = base_url.rstrip("/") + "/" + file_name
    local_path = os.path.join(year_folder, file_name)

    try:
      res = requests.get(file_url, headers=headers, verify=False, timeout=20)

      if res.status_code == 200:
        with open(local_path, "wb") as f:
          f.write(res.content)
        print(f"Año {year_str} | Mes {mes_str}: Descargado con éxito")
      else:
        print(
            f"Año {year_str} | Mes {mes_str}: No disponible (Status"
            f" {res.status_code})"
        )

      # Pausa de cortesía para no saturar al servidor
      time.sleep(0.5)

    except Exception as e:
      print(f"Error al descargar {file_name}: {e}")

print("\nDescargas de 2019 a 2023 completadas.")

----------------------------------------
Descarga del año: 2019
----------------------------------------
Año 2019 | Mes 01: Descargado con éxito
Año 2019 | Mes 02: Descargado con éxito
Año 2019 | Mes 03: Descargado con éxito
Año 2019 | Mes 04: Descargado con éxito
Año 2019 | Mes 05: Descargado con éxito
Año 2019 | Mes 06: Descargado con éxito
Año 2019 | Mes 07: Descargado con éxito
Año 2019 | Mes 08: Descargado con éxito
Año 2019 | Mes 09: Descargado con éxito
Año 2019 | Mes 10: Descargado con éxito
Año 2019 | Mes 11: Descargado con éxito
Año 2019 | Mes 12: Descargado con éxito
----------------------------------------
Descarga del año: 2020
----------------------------------------
Año 2020 | Mes 01: Descargado con éxito
Año 2020 | Mes 02: Descargado con éxito
Año 2020 | Mes 03: Descargado con éxito
Año 2020 | Mes 04: Descargado con éxito
Año 2020 | Mes 05: Descargado con éxito
Año 2020 | Mes 06: Descargado con éxito
Año 2020 | Mes 07: Descargado con éxito
Año 2020 | Mes 08: Descargado 

## Procesamiento y Limpieza

In [ ]:
import glob
import os
import pandas as pd

# 1. Definir la ruta hacia los datos crudos desde la carpeta 'notebooks'
# Buscamos recursivamente todos los .csv dentro de las carpetas de lluvia
patron = os.path.join("..", "data", "raw", "temperatura_*", "*.csv")

# Definir rutas absolutas con glob.glob
archivos = sorted(glob.glob(patron))
print(f"Total de archivos CSV encontrados: {len(archivos)}")

# Lista que nos servira para a;adir en cada ciclo un archivo csv
lista_nacional = []

# 2. Procesar cada archivo mensual
for ruta in archivos:
  try:
    df_mes = pd.read_csv(ruta, encoding="latin-1")
    df_mes.columns = [c.strip().lower() for c in df_mes.columns] # Limpiar y normalizar nombres de columnas a minúsculas    
    df_mes["archivo_origen"] = os.path.basename(ruta) # Añadir una columna con el nombre del archivo original
    lista_nacional.append(df_mes) # Añadir a la lista el df acctual
  except Exception as e:
    print(f"Error al leer {ruta}: {e}")

  
# 3. Concatenar los df
if lista_nacional:
    df_temperaturas_nacional = pd.concat(lista_nacional, ignore_index=True) # Concatenar todas las listas
    print(
        f"¡Dataset nacional consolidado! Filas: {len(df_temperaturas_nacional):,} |"
        f" Columnas: {df_temperaturas_nacional.shape[1]}")
    
# limpieza y tipificación de temporal del dataframe 
#Cambiar el nombre de la columna temperatura
df_temperaturas_nacional.rename(columns={'tmed':'temp_promedio'}, inplace=True)


# Creacion de columnas Fecha, Año, y Mes
df_temperaturas_nacional['anio'] = df_temperaturas_nacional['archivo_origen'].str[:4].astype(int)
df_temperaturas_nacional['mes'] = df_temperaturas_nacional['archivo_origen'].str[4:6].astype(int)

# Crear una columan de fecha con los valores de anio y mes
anio = df_temperaturas_nacional['archivo_origen'].str[:4].astype(str)
mes = df_temperaturas_nacional['archivo_origen'].str[4:6].astype(str)

df_temperaturas_nacional['fecha'] = pd.to_datetime(anio + '-' + mes + '-' + '01')
df_temperaturas_nacional.head()



Total de archivos CSV encontrados: 60
¡Dataset nacional consolidado! Filas: 75,892 | Columnas: 7


,lon,lat,clave,edo,est,temp_promedio,archivo_origen,anio,mes,fecha
0,-99.75,16.76,76805,GRO,ACAPULCO,27.441935,201901010000TMed.csv,2019,1,2019-01-01
1,-111.83,30.71,76113,SON,ALTAR,13.462069,201901010000TMed.csv,2019,1,2019-01-01
2,-93.90,16.24,76840,CHIS,ARRIAGA,28.088710,201901010000TMed.csv,2019,1,2019-01-01
3,-90.50,19.83,76695,CAMP,CAMPECHE,23.374074,201901010000TMed.csv,2019,1,2019-01-01
4,-111.65,25.01,76402,BCS,CD. CONSTITUCIÓN,17.987097,201901010000TMed.csv,2019,1,2019-01-01


In [20]:

# Guardar el archivo consolidado
# Confirmar directorio
print(os.getcwd())

# Ruta de salida
ruta_salida = os.path.join("..", "data", "processed", "temperaturas_nacional_2019_2023.csv")

# Crear df en la ruta de salida
df_temperaturas_nacional.to_csv(ruta_salida, index=False, encoding='utf-8-sig')

c:\Users\victo\OneDrive\Documentos Personales\GitHub\sonora-agriclimate-eda\notebooks


# Validación de datos 
### Schema basándonos en el diccionario de datos

In [24]:
from datetime import date, datetime
from typing import Optional
import numpy as np
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, field_validator


class TemperaturaEstacionSchema(BaseModel):
  model_config = ConfigDict(extra="ignore")

  # Variables de ubicación geográfica
  lon: Optional[float] = Field(default=None, ge=-180.0, le=180.0)
  lat: Optional[float] = Field(default=None, ge=-90.0, le=90.0)

  # Variables de identificación de estación y estado
  clave: Optional[str] = None
  edo: Optional[str] = None
  est: Optional[str] = None

  # Medición meteorológica (temperatura media mensual en °C)
  temp_promedio: Optional[float] = Field(default=None, ge=-50.0, le=65.0)

  # Metadatos del archivo origen
  archivo_origen: Optional[str] = None

  # Variables temporales derivadas
  anio: Optional[int] = Field(default=None, ge=1900, le=2100)
  mes: Optional[int] = Field(default=None, ge=1, le=12)
  fecha: Optional[date] = None

  # 1. Normalizador universal de nulos y valores centinela del SMN (-99.9, -999)
  @field_validator("*", mode="before")
  @classmethod
  def normalize_missing_and_sentinel_values(cls, value):
    if value is None or pd.isna(value):
      return None

    # Si es texto y representa vacío o nan
    if str(value).strip().lower() in ["", "nan", "none", "null"]:
      return None

    # Detectar códigos centinela comunes de CONAGUA/SMN
    try:
      val_num = float(value)
      if val_num in [-99.9, -999.0, -99.0]:
        return None
    except (ValueError, TypeError):
      pass

    return value

  # 2. Limpieza de campos de texto
  @field_validator(
      "clave", "edo", "est", "archivo_origen", mode="before"
  )
  @classmethod
  def convert_text_fields(cls, value):
    if value is None or pd.isna(value):
      return None
    return str(value).strip()

  # 3. Conversión flexible de fechas
  @field_validator("fecha", mode="before")
  @classmethod
  def parse_date(cls, value):
    if value is None or pd.isna(value):
      return None
    if isinstance(value, pd.Timestamp):
      return value.date()
    if isinstance(value, datetime):
      return value.date()
    if isinstance(value, str):
      return datetime.strptime(value.strip()[:10], "%Y-%m-%d").date()
    return value

### Función para validar los datos descargados

In [25]:
from pydantic import ValidationError
import pandas as pd

def validate_df(df: pd.DataFrame) -> pd.DataFrame:
    validated_rows = []

    for index, row in df.iterrows():
        try:
            temperature = TemperaturaEstacionSchema(**row.to_dict())
            validated_rows.append(temperature.model_dump())

        except ValidationError as e:
            print(f"Error en fila {index}:")
            print(e)

    return pd.DataFrame(validated_rows)

# Validar el DataFrame consolidado de temperaturas
df_temperaturas_validadas = validate_df(df_temperaturas_nacional)
df_temperaturas_validadas.head(5)

,lon,lat,clave,edo,est,temp_promedio,archivo_origen,anio,mes,fecha
0,-99.75,16.76,76805,GRO,ACAPULCO,27.441935,201901010000TMed.csv,2019,1,2019-01-01
1,-111.83,30.71,76113,SON,ALTAR,13.462069,201901010000TMed.csv,2019,1,2019-01-01
2,-93.90,16.24,76840,CHIS,ARRIAGA,28.088710,201901010000TMed.csv,2019,1,2019-01-01
3,-90.50,19.83,76695,CAMP,CAMPECHE,23.374074,201901010000TMed.csv,2019,1,2019-01-01
4,-111.65,25.01,76402,BCS,CD. CONSTITUCIÓN,17.987097,201901010000TMed.csv,2019,1,2019-01-01


In [26]:
# 3. Guardar el archivo final
ruta_salida = os.path.join(
    "..", "data", "processed", "temperaturas_nacional_2019_2023.csv"
)
df_temperaturas_validadas.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"Archivo exportado exitosamente a: {ruta_salida}")

Archivo exportado exitosamente a: ..\data\processed\temperaturas_nacional_2019_2023.csv
